## 1. Import Libraries

In [17]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Read CSV Files

In [18]:
job_vacancies_by_industry = pd.read_csv("Data/job_vacancies_by_industry.csv")

## 3. Show First 5 Rows Of Dataset

In [19]:
job_vacancies_by_industry.head()

,year,industry,occupation,job_vacancy_rate
0,1998,manufacturing,"professionals, managers, executives and techni...",1.8
1,1998,manufacturing,"clerical, sales and services workers",1.2
2,1998,manufacturing,"production and transport operators, cleaners a...",1.6
3,1998,"food, beverages and tobacco","professionals, managers, executives and techni...",-
4,1998,"food, beverages and tobacco","clerical, sales and services workers",1.3


# 4. Data Cleaning

## 4.A. Copy of job_vacancies_by_industry for cleaning

In [20]:
job_vacancies_by_industry_clean = job_vacancies_by_industry.copy()
print("Shape:", job_vacancies_by_industry_clean.shape)

Shape: (4554, 4)


## 4.B. Dataset summary before cleaning

In [21]:
column_summary = pd.DataFrame({
    "data_type": job_vacancies_by_industry_clean.dtypes.astype(str),
    "missing_count": job_vacancies_by_industry_clean.isna().sum(),
    "missing_percentage": (job_vacancies_by_industry_clean.isna().mean() * 100).round(2),
    "unique_count": job_vacancies_by_industry_clean.nunique(dropna=False)
})

display(column_summary)

print("Empty Rows:", job_vacancies_by_industry_clean.isna().all(axis=1).sum())
print("Exact duplicate rows:", job_vacancies_by_industry_clean.duplicated().sum())
print("Duplicate identifier records:", job_vacancies_by_industry_clean.duplicated(subset=["year", "industry", "occupation"]).sum())

,data_type,missing_count,missing_percentage,unique_count
year,int64,0,0.0,28
industry,object,0,0.0,68
occupation,object,0,0.0,5
job_vacancy_rate,object,0,0.0,109


Empty Rows: 0
Exact duplicate rows: 0
Duplicate identifier records: 0


## 4.C. Checking for placeholder values

In [22]:
placeholder_values = {"", "na", "n/a", "null", "none", "unknown"}

placeholder_report = []

for column in ["industry", "occupation"]:
    normalised_values = (
        job_vacancies_by_industry_clean[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    count = normalised_values.isin(placeholder_values).sum()

    placeholder_report.append({"Column": column, "Placeholder Count": count})

display(pd.DataFrame(placeholder_report))

,Column,Placeholder Count
0,industry,0
1,occupation,0


## 4.D. Convert placeholders to NaN

In [23]:
for column in ["industry", "occupation"]:
    normalised_values = (
        job_vacancies_by_industry_clean[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    placeholder_mask = normalised_values.isin(placeholder_values)
    job_vacancies_by_industry_clean.loc[placeholder_mask, column] = np.nan

print("Missing values after replacing placeholders:")
display(job_vacancies_by_industry_clean.isna().sum())

Missing values after replacing placeholders:


year                0
industry            0
occupation          0
job_vacancy_rate    0
dtype: int64

## 4.E. Check for duplicate records

In [24]:
duplicate_rows = job_vacancies_by_industry_clean.duplicated().sum()
print("Duplicate rows before removal:", duplicate_rows)

Duplicate rows before removal: 0


## 4.F. Remove duplicates

In [25]:
rows_before = len(job_vacancies_by_industry_clean)

job_vacancies_by_industry_clean = job_vacancies_by_industry_clean.drop_duplicates()
job_vacancies_by_industry_clean.reset_index(drop=True, inplace=True)

rows_removed = rows_before - len(job_vacancies_by_industry_clean)
print("Rows removed:", rows_removed)

Rows removed: 0


## 4.G. Validate year column, remove invalid years

In [26]:
year_numeric_test = pd.to_numeric(job_vacancies_by_industry_clean["year"], errors="coerce")

invalid_year_mask = year_numeric_test.isna() & job_vacancies_by_industry_clean["year"].notna()
print("Rows containing invalid year values:", invalid_year_mask.sum())
display(job_vacancies_by_industry_clean.loc[invalid_year_mask])

rows_before = len(job_vacancies_by_industry_clean)
job_vacancies_by_industry_clean = job_vacancies_by_industry_clean.loc[~invalid_year_mask].copy()
rows_removed = rows_before - len(job_vacancies_by_industry_clean)
print("Rows removed due to invalid years:", rows_removed)

Rows containing invalid year values: 0


,year,industry,occupation,job_vacancy_rate


Rows removed due to invalid years: 0


## 4.H. Convert year to integer, check distribution

In [27]:
job_vacancies_by_industry_clean["year"] = (
    pd.to_numeric(job_vacancies_by_industry_clean["year"], errors="coerce").astype("Int64")
)

print(job_vacancies_by_industry_clean["year"].dtype)

display(job_vacancies_by_industry_clean["year"].value_counts(dropna=False).sort_index())

Int64


year
1998     84
1999     84
2000    129
2001    129
2002    129
2003    129
2004    129
2005    129
2006    126
2007    126
2008    126
2009    126
2010    126
2011    126
2012    126
2013    210
2014    210
2015    210
2016    210
2017    210
2018    210
2019    210
2020    210
2021    210
2022    210
2023    210
2024    210
2025    210
Name: count, dtype: Int64

## 4.I. Check consistency for industry and occupation

In [28]:
display(job_vacancies_by_industry_clean["industry"].value_counts(dropna=False))
display(job_vacancies_by_industry_clean["occupation"].value_counts(dropna=False))

industry
manufacturing                              110
community, social and personal services    110
food, beverages and tobacco                110
retail trade                               110
wholesale trade                            110
                                          ... 
paper products and publishing                6
business and real estate services            6
transport, storage and communications        6
financial intermediation                     6
petroleum and chemical products              6
Name: count, Length: 68, dtype: int64

occupation
professionals, managers, executives and technicians           1154
clerical, sales and services workers                          1154
production and transport operators, cleaners and labourers    1154
professionals, managers and executives                         546
non-professionals, managers and executives                     546
Name: count, dtype: int64

## 4.J. Standardise text columns

In [29]:
text_columns = ["industry", "occupation"]

for column in text_columns:
    job_vacancies_by_industry_clean[column] = (
        job_vacancies_by_industry_clean[column]
        .astype("string")
        .str.strip()
        .str.title()
    )

## 4.K. Verify standardised categories

In [30]:
print("Unique values in Industry:")
display(sorted(job_vacancies_by_industry_clean["industry"].dropna().unique()))

print("Unique values in Occupation:")
display(sorted(job_vacancies_by_industry_clean["occupation"].dropna().unique()))

Unique values in Industry:


['Accommodation',
 'Accommodation And Food Services',
 'Administrative And Support Services',
 'Air Transport And Supporting Services',
 'Architectural And Engineering Services',
 'Arts, Entertainment And Recreation',
 'Broadcasting And Publishing',
 'Business And Real Estate Services',
 'Cleaning And Landscaping',
 'Community, Social And Personal Services',
 'Construction',
 'Education',
 'Electrical Products',
 'Electronic Products',
 'Electronic, Computer And Optical Products',
 'Fabricated Metal Products',
 'Fabricated Metal Products, Machinery And Equipment',
 'Financial And Insurance Services',
 'Financial Institutions',
 'Financial Intermediation',
 'Financial Services',
 'Food And Beverage Services',
 'Food, Beverages And Tobacco',
 'Health And Social Services',
 'Health, Social And Community Services',
 'Hotels',
 'Hotels And Restaurants',
 'Information And Communications',
 'Insurance',
 'Insurance And Pension Funding',
 'Insurance Services',
 'It And Other Information Servic

Unique values in Occupation:


['Clerical, Sales And Services Workers',
 'Non-Professionals, Managers And Executives',
 'Production And Transport Operators, Cleaners And Labourers',
 'Professionals, Managers And Executives',
 'Professionals, Managers, Executives And Technicians']

## 4.L. Convert job_vacancy_rate to numeric, check negatives

In [31]:
job_vacancies_by_industry_clean["job_vacancy_rate"] = pd.to_numeric(
    job_vacancies_by_industry_clean["job_vacancy_rate"], errors="coerce"
)

print(job_vacancies_by_industry_clean["job_vacancy_rate"].dtype)

negative_values = (job_vacancies_by_industry_clean["job_vacancy_rate"] < 0).sum()
print("Negative job vacancy rate values:", negative_values)

float64
Negative job vacancy rate values: 0


## 4.M. Summary after cleaning

In [32]:
print("Final Shape:")
print(job_vacancies_by_industry_clean.shape)

print("\nMissing Values:")
display(job_vacancies_by_industry_clean.isna().sum())

print("\nDuplicate Rows:")
print(job_vacancies_by_industry_clean.duplicated().sum())

print("\nData Types:")
display(job_vacancies_by_industry_clean.dtypes)

Final Shape:
(4554, 4)

Missing Values:


year                  0
industry              0
occupation            0
job_vacancy_rate    546
dtype: int64


Duplicate Rows:
0

Data Types:


year                         Int64
industry            string[python]
occupation          string[python]
job_vacancy_rate           float64
dtype: object